In [1]:
import torch
import pado
from pado.math import mm, um, nm

In [2]:
# 1) Parameters:
R, C = 2048, 2048                # array size (increase to reduce aliasing)
pitch = 6.4 * um                 # SLM pixel pitch in meters
wvl = 532 * nm                   # wavelength (single-channel example)
dim = (1, 1, R, C)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
# 2) Create propagator
prop = pado.propagator.Propagator('ASM')

# 3) distances for target planes (in meters)
depths_mm = [60, 100, 150, 200, 250]                # mm
depths = [d * mm for d in depths_mm]      # convert with pado.math.mm

# 4) Load/build an image for each depth, create Light objects
lights = []
for z in enumerate(depths):
    L = pado.light.Light(dim, pitch, wvl, device=device)
    # load image; random_phase for amplitude-only inputs
    img_path = f'../example/asset/bunny.PNG'
    L.load_image(image_path=img_path, random_phase=True)
    lights.append((L, z))

In [4]:
# 5) Propagate each target plane to the SLM plane (positive z moves toward SLM)
slm_fields = []
for L, z in lights:
    # linear=True pads for linear convolution; band_limit=True avoids aliasing
    L_at_slm = prop.forward(L, z, linear=True, band_limit=True)
    slm_fields.append(L_at_slm.get_field())  # complex tensor [B,Ch,H,W]

# 6) Sum complex fields at SLM plane (superposition)
total_field = sum(slm_fields)  # careful: ensure shapes & devices match

RuntimeError: The size of tensor a (4096) must match the size of tensor b (2) at non-singleton dimension 3

In [ ]:
# 7) Make phase-only SLM pattern: extract phase, map to 0..2pi
slm_phase = torch.angle(total_field)        # in [-pi, pi]
slm_phase = (slm_phase + 2*torch.pi) % (2*torch.pi)  # convert to [0,2pi)
# If SLM expects 8-bit image:
slm_img = (slm_phase / (2*torch.pi) * 255.0).clamp(0,255).to(torch.uint8)
# Save or visualize: use existing Light helpers or torchvision
# Example using pillow:
from PIL import Image
img_np = slm_img.squeeze().cpu().numpy()
Image.fromarray(img_np).save('slm_phase_3depths.png')